# Loading and Merging Snapshot Data from Varaosahaku.fi

## 1) Imports

In [ ]:
from pathlib import Path
import pandas as pd

## 2) Collect Source CSV Files

In [ ]:
root = Path.cwd() if (Path.cwd() / 'README.md').exists() else Path.cwd().parent
all_csv_files = []
for csv_file in sorted(root.rglob('*.csv')):
    rel = str(csv_file.relative_to(root)).lower()
    if rel.startswith('datasets/merged/'):
        continue
    all_csv_files.append(csv_file)

## 3) Keep Only Files with Valid Core Columns

In [ ]:
required_cols = [
    'part_name', 'price', 'quality_grade', 'year', 'oem_number',
    'engine_code', 'mileage', 'brand', 'model', 'category', 'subcategory'
]

valid_csv_files = []
for csv_file in all_csv_files:
    header_df = pd.read_csv(csv_file, nrows=0)
    cols = set(header_df.columns)
    if all(c in cols for c in required_cols):
        valid_csv_files.append(csv_file)

## 4) Group Files by Car Brand and Model

In [ ]:
golf_files = []
skoda_files = []
toyota_files = []

for csv_file in valid_csv_files:
    text = str(csv_file).lower()
    if 'golf' in text or 'e_golf' in text:
        golf_files.append(csv_file)
        continue
    if 'skoda' in text or 'octavia' in text:
        skoda_files.append(csv_file)
        continue
    if 'toyota' in text or 'corolla' in text or 'yaris' in text:
        toyota_files.append(csv_file)

## 5) Merge Each Group

In [ ]:
output_cols = [
    'product_id', 'part_name', 'price', 'quality_grade', 'year', 'oem_number',
    'engine_code', 'mileage', 'brand', 'model', 'category', 'subcategory',
    'scrape_date', 'scrape_timestamp'
]

def merge_files(files):
    frames = []
    for csv_file in files:
        df = pd.read_csv(csv_file)
        if 'roduct_id' in df.columns and 'product_id' not in df.columns:
            df = df.rename(columns={'roduct_id': 'product_id'})
        df = df.reindex(columns=output_cols)
        frames.append(df)
    return pd.concat(frames, ignore_index=True, sort=False)

golf_df = merge_files(golf_files)
skoda_df = merge_files(skoda_files)
toyota_df = merge_files(toyota_files)

## 6) Save Files

In [ ]:
out_dir = root / 'datasets' / 'merged'
out_dir.mkdir(parents=True, exist_ok=True)

golf_df.to_csv(out_dir / 'dppm_vw_golf_e_golf.csv', index=False)
skoda_df.to_csv(out_dir / 'dppm_skoda_octavia.csv', index=False)
toyota_df.to_csv(out_dir / 'dppm_toyota_corolla_yaris.csv', index=False)